# Laad de data in duckdb en verbeter de datakwaliteit volgens de DNA-Nl datakwaliteitsrichtlijnen voor laag "zilver"

Auteurs: Niels Molenaar, Maurice de Kleijn, Thya van den Berg

Organisatie: Rijksdienst voor het cultureel erfgoed

Programma: DNA-NL

Project: Bronnen- en datakwaliteit

Projectleider: Maurice de Kleijn


In deze notebook wordt de datakwaliteit van de RCE bronnen gemeten en mogelijke verbeteringen geidentificeerd.

Per dataset weergegeven aan hoeveel (x/9), en welke datakwaliteitsdimensies deze "out of the box" al dan niet voldoet.

Mogelijke verbeteringen worden ingedeeld op 3 niveaus:
* Quick wins: verbeteringen die met een klik op de knop, enkele query of kort stukje code gemakkelijk kunnen worden aangepast.
* Medium wins: verbeteringen die geautomatiseerd kunnen worden, maar wel meer werk of technische kunde vereisen, zoals fuzzy matching of het opzetten van een (behapbare) draaitabel.
* Handmatig: verbeteringen die bijna per rij moeten worden toegepast en slecht te automatiseren zijn, waardoor deze het uitpluizen door een mens (of bij AI oplossingen een human-in-the-loop approach) vereisen.

Alleen de quick wins worden hier ter demonstratie ook gelijk toegepast.

## installeer packages

In [1]:
# %pip install duckdb
# %pip install matplotlib
# %pip install mpl_toolkits
# %pip install shapely

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement mpl_toolkits (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for mpl_toolkits


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## laad packages

In [2]:
import duckdb
import geopandas as gpd
import pandas as pd
import os
import matplotlib.pyplot as plt
from shapely import wkt


## Benodigde variabelen

### Variabelen definiëren voor Medallion Architecture


In [3]:
datalake_path = os.path.normpath("./data/")
zilveren_laag_path = "zilver"

# voor tussendoor opslaan
save_directory = os.path.normpath(r"./data/saved_parquet_files//")
if not os.path.exists(save_directory):
    # maak directory als deze nog niet bestaat
    os.mkdir(save_directory)

# voor opslaan "quick wins"
zilver_directory = os.path.normpath(f"./data/{zilveren_laag_path}//")
if not os.path.exists(zilver_directory):
    # maak directory als deze nog niet bestaat
    os.mkdir(zilver_directory)

## Benodigde functies

In [ ]:
# sla output van een query op (template)
# file_name = 'query_output.parquet'
# duckdb.sql(f"""
#             COPY
#                 (SELECT x.*, y.*
#                 FROM
#                 brons_beschermde_stads_en_dorpsgezichten.townscapes x
#                 JOIN
#                 brons_rijksmonumentenregister.tblTEXT_OBJECT y
#                 ON
#                 x.bron_id = y.OBJ_NUMMER)
#                 TO '{os.path.join(save_directory, file_name)}'
#                 (FORMAT parquet);""")
#
# # alternatief
# duckdb.sql(f"""SELECT x.*, y.*
#                 FROM
#                 brons_beschermde_stads_en_dorpsgezichten.townscapes x
#                 JOIN
#                 brons_rijksmonumentenregister.tblTEXT_OBJECT y
#                 ON
#                 x.bron_id = y.OBJ_NUMMER""").write_parquet(os.path.join(save_directory, file_name))

def empty_duckdb_memory():
    """Empty the in-memory duckdb database"""
    schemas = duckdb.sql("""
                SELECT schema_name
                FROM information_schema.schemata
                WHERE schema_name NOT IN ('information_schema', 'pg_catalog', 'temp', 'main')""").fetchdf()
    for schema in schemas['schema_name']:
        tables = duckdb.sql(f"""
                        SELECT table_name
                        FROM information_schema.tables
                        WHERE table_schema = '{schema}'
        """).fetchdf()
        for table in tables['table_name']:
            duckdb.sql(f"DROP TABLE IF EXISTS {schema}.{table}")
        duckdb.sql(f"DROP SCHEMA IF EXISTS {schema}")


def save_complete_duckdb_memory(save_directory):
    """saves entire in-memory duckdb database to the given save_directory"""
    duckdb.sql(f"""EXPORT DATABASE '{save_directory}' (FORMAT parquet)""")

    def load_duckdbset_to_memory(load_directory):
        """Loads an entire saved set of parquet files from the included "schema.sql" and "load.sql" files."""

    file_name = 'schema.sql'
    with open(os.path.join(load_directory, file_name), 'r') as schema:
        duckdb.sql(schema.read())

    file_name = 'load.sql'
    with open(os.path.join(load_directory, file_name), 'r') as load:
        duckdb.sql(load.read())
    duckdb.sql("SHOW SCHEMAS;")

## Setup van duckdb

duckdb is een databasemanagementsoftware vergelijkbaar met PostGRES en MySQL. Het gebruikt SQL queries om met de data te interacteren. In dit notebook gaat dit volledig lokaal en in het werkgeheugen van de computer, anders dan in het DAP, echter, de queries die moeten worden uitgevoerd voor het verbeteren van de data zijn hetzelfde.

### Laad de Spatial Extension
https://duckdb.org/docs/current/core_extensions/spatial/overview

In [4]:
duckdb.sql("""
           INSTALL spatial;
           LOAD spatial;
           """)

### laad databases (schema's) in de bronzen laag

In [5]:
duckdb.sql("""
           CREATE SCHEMA IF NOT EXISTS brons_beschermde_stads_en_dorpsgezichten;
           CREATE SCHEMA IF NOT EXISTS brons_bestuurlijke_gebieden_2026;
           CREATE SCHEMA IF NOT EXISTS brons_publiekrechtelijke_beperkingen;
           CREATE SCHEMA IF NOT EXISTS brons_rijksbeschermde_groenaanleggen;
           CREATE SCHEMA IF NOT EXISTS brons_rijksmonumentale_boerderijen;
           CREATE SCHEMA IF NOT EXISTS brons_beschermde_stads_en_dorpsgezichten;
           CREATE SCHEMA IF NOT EXISTS brons_rijksmonumentale_sluizen_en_stuwen;
           CREATE SCHEMA IF NOT EXISTS brons_rijksmonumentenpunten;
           CREATE SCHEMA IF NOT EXISTS brons_rijksmonumentencontouren;
           CREATE SCHEMA IF NOT EXISTS brons_rijksmonumentenregister;
           CREATE SCHEMA IF NOT EXISTS brons_basisregistratie_adressen_en_gebouwen;
           """)

### laad parquet files als een tabellen in de bronzen laag

In [8]:
# beschermde_stads_en_dorpsgezichten in brons
try:
    duckdb.sql("""
           CREATE TABLE IF NOT EXISTS brons_beschermde_stads_en_dorpsgezichten.townscapes AS
           FROM read_parquet('./data/brons/beschermde_stads_en_dorpsgezichten/townscapes.parquet');
           """)
except Exception as e:
    print(e, " skipping stads_en_dorpsgezichten")
# bestuurlijke_gebieden_2026 in brons
try:
    duckdb.sql("""
           CREATE TABLE IF NOT EXISTS brons_bestuurlijke_gebieden_2026.gemeentegebied AS
           FROM read_parquet('./data/brons/bestuurlijke_gebieden_2026/gemeentegebied.parquet');
           CREATE TABLE IF NOT EXISTS brons_bestuurlijke_gebieden_2026.landgebied AS
           FROM read_parquet('./data/brons/bestuurlijke_gebieden_2026/landgebied.parquet');
           CREATE TABLE IF NOT EXISTS brons_bestuurlijke_gebieden_2026.provinciegebied AS
           FROM read_parquet('./data/brons/bestuurlijke_gebieden_2026/provinciegebied.parquet');
           """)
except Exception as e:
    print(e, " skipping bestuurlijke gebieden 2026")
# publiekrechtelijke_beperkingen in brons
try:
    duckdb.sql("""
               CREATE TABLE IF NOT EXISTS brons_publiekrechtelijke_beperkingen.pb_pb_multilinestring AS
               FROM read_parquet('./data/brons/publiekrechtelijke_beperkingen/pb_multipolygon.parquet');
               CREATE TABLE IF NOT EXISTS brons_publiekrechtelijke_beperkingen.pb_pb_multipolygon AS
               FROM read_parquet('./data/brons/publiekrechtelijke_beperkingen/pb_multipolygon.parquet');
               """)
except Exception as e:
    print(e, " skipping publieksgerechtelijke bescherming")
# rijksbeschermde_groenaanleggen in brons
try:
    duckdb.sql("""
               CREATE TABLE IF NOT EXISTS brons_rijksbeschermde_groenaanleggen.rijksmonumentaal_groen AS
               FROM read_parquet('./data/brons/rijksbeschermde_groenaanleggen/rijksmonumentaal_groen.parquet');
               """)
except Exception as e:
    print(e, " skipping rijksmonumentaal groen")
# rijksmonumentale_boerderijen in brons
try:
    duckdb.sql("""
           CREATE TABLE IF NOT EXISTS brons_rijksmonumentale_boerderijen.boerderijen AS
           FROM read_parquet('./data/brons/rijksmonumentale_boerderijen/boerderijen.parquet');
           """)
except Exception as e:
    print(e, " skipping rijksmonumentale boerderijen")
# rijksmonumentale_sluizen_en_stuwen in brons
try:
    duckdb.sql("""
               CREATE TABLE IF NOT EXISTS brons_rijksmonumentale_sluizen_en_stuwen.sluizen_stuwen AS
               FROM read_parquet('./data/brons/rijksmonumentale_sluizen_en_stuwen/sluizen_stuwen.parquet');
               """)
except Exception as e:
    print(e, " skipping rijksmonumentale sluizen_stuwen")
# rijksmonumentenpunten in brons
try:
    duckdb.sql("""
            CREATE TABLE IF NOT EXISTS brons_rijksmonumentenpunten.rijksmonumentenpunten AS FROM read_parquet('./data/brons/rijksmonumentenpunten/rijksmonumentpunten.parquet');
            """)
except Exception as e:
    print(e, " skipping rijksmonumentenpunten")

# rijksmonumentencontouren in brons
try:
    duckdb.sql("""
            CREATE TABLE IF NOT EXISTS brons_rijksmonumentencontouren.rijksmonumentencontouren AS FROM read_parquet('./data/brons/rijksmonumentencontouren/rijksmonumentcontouren.parquet');
            """)
except Exception as e:
    print(e, " skipping rijksmonumentencontouren")
# rijksmonumentenregister in brons
try:
    duckdb.sql("""
           CREATE TABLE IF NOT EXISTS brons_rijksmonumentenregister.tblTEXT_OBJECT AS
           FROM read_parquet('./data/brons/rijksmonumentenregister/tblTEXT_OBJECT.parquet');
           """) # TODO aanvullen met rest relevanten tables
except Exception as e:
    print(e, " skipping rms")
# BAG in brons
try:
    duckdb.sql("""
            CREATE TABLE IF NOT EXISTS brons_basisregistratie_adressen_en_gebouwen.pand AS FROM read_parquet(
            './data/brons/basisregistratie_adressen_en_gebouwen/pand.parquet');
            CREATE TABLE IF NOT EXISTS brons_basisregistratie_adressen_en_gebouwen.verblijfsobject AS FROM read_parquet(
            './data/brons/basisregistratie_adressen_en_gebouwen/verblijfsobject.parquet');
            CREATE TABLE IF NOT EXISTS brons_basisregistratie_adressen_en_gebouwen.woonplaats AS FROM read_parquet(
            './data/brons/basisregistratie_adressen_en_gebouwen/woonplaats.parquet');
            """)
except Exception as e:
    print(e, " skipping bag")



IO Error: No files found that match the pattern "./data/brons/bestuurlijke_gebieden_2026/gemeentegebied.parquet"

LINE 3:            FROM read_parquet('./data/brons/bestuurlijke_gebieden_2026/gemee...
                        ^  skipping bestuurlijke gebieden 2026
IO Error: No files found that match the pattern "./data/brons/publiekrechtelijke_beperkingen/pb_multipolygon.parquet"

LINE 3:                FROM read_parquet('./data/brons/publiekrechtelijke_beperkingen...
                            ^  skipping publieksgerechtelijke bescherming
IO Error: No files found that match the pattern "./data/brons/basisregistratie_adressen_en_gebouwen/pand.parquet"

LINE 2: ... brons_basisregistratie_adressen_en_gebouwen.pand AS FROM read_parquet(
                                                                     ^  skipping bag


## Beschrijf de tabellen in de bronzen laag


### stads en dorpsgezichten

In [9]:
# beschermde_stads_en_dorpsgezichten in brons
duckdb.sql("""
           DESCRIBE  brons_beschermde_stads_en_dorpsgezichten.townscapes;
           """)


┌─────────────┬────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │      column_type       │  null   │   key   │ default │  extra  │
│   varchar   │        varchar         │ varchar │ varchar │ varchar │ varchar │
├─────────────┼────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ bron_id     │ VARCHAR                │ YES     │ NULL    │ NULL    │ NULL    │
│ naam        │ VARCHAR                │ YES     │ NULL    │ NULL    │ NULL    │
│ in_procedu  │ TIMESTAMP              │ YES     │ NULL    │ NULL    │ NULL    │
│ aangewezen  │ TIMESTAMP              │ YES     │ NULL    │ NULL    │ NULL    │
│ jurstatus   │ VARCHAR                │ YES     │ NULL    │ NULL    │ NULL    │
│ datum_begr  │ TIMESTAMP              │ YES     │ NULL    │ NULL    │ NULL    │
│ ondergrond  │ VARCHAR                │ YES     │ NULL    │ NULL    │ NULL    │
│ opperv_ha   │ DOUBLE                 │ YES     │ NULL    │ NULL    │ NULL    │
│ opperv_km2  │ DOUBLE      

### gemeentegebied

In [10]:
duckdb.sql("""
           DESCRIBE  brons_bestuurlijke_gebieden_2026.gemeentegebied;
           """)

CatalogException: Catalog Error: Table with name gemeentegebied does not exist!
Did you mean "pg_catalog.pg_depend"?

### landgebied

In [ ]:
duckdb.sql("""
           DESCRIBE  brons_bestuurlijke_gebieden_2026.landgebied;
           """)

### Provinciegebied

In [ ]:
duckdb.sql("""
           DESCRIBE  brons_bestuurlijke_gebieden_2026.gemeentegebied;
           """)

### Publieksrechtelijke bescherming lijn

In [ ]:
duckdb.sql("""
           DESCRIBE  brons_publiekrechtelijke_beperkingen.pb_pb_multilinestring;
           """)

### Publieksrechtelijke bescherming vlak

In [ ]:
duckdb.sql("""
           DESCRIBE  brons_publiekrechtelijke_beperkingen.pb_pb_multipolygon;
           """)

### Rijksmonumentaal groen

In [ ]:
duckdb.sql("""
           DESCRIBE  brons_rijksbeschermde_groenaanleggen.rijksmonumentaal_groen;
           """)

### rijskmonumentable boerderijen

In [ ]:
duckdb.sql("""
           DESCRIBE  brons_rijksmonumentale_boerderijen.boerderijen;
           """)

### Stuwen en Sluizen

In [ ]:
duckdb.sql("""
           DESCRIBE  brons_rijksmonumentale_sluizen_en_stuwen.sluizen_stuwen;
           """)

### Rijksmonumentenpunten

In [ ]:
duckdb.sql("""
           DESCRIBE  brons_rijksmonumentenpunten.rijksmonumentenpunten;
           """)

### Rijksmonumentencontouren

In [ ]:
duckdb.sql("""
           DESCRIBE  brons_rijksmonumentencontouren.rijksmonumentencontouren;
           """)

### tblTEXT_OBJECT (rms)

In [ ]:
duckdb.sql("""
           DESCRIBE  brons_rijksmonumentenregister.tblTEXT_OBJECT;
           """)

### nog meer? (template)

In [ ]:
duckdb.sql("""
           DESCRIBE  brons_voorbeeld.voorbeeld;
           """)

## Testjes

Selecteer de geometrie en laat zien.

In [ ]:
df= duckdb.sql("""SELECT ST_asText(geometry) FROM brons_beschermde_stads_en_dorpsgezichten.townscapes;
           """).fetchdf()
df

join twee databases met elkaar op rijksmonumentnummer

In [ ]:
df = duckdb.sql(
    """
    SELECT x.*, y.*
    FROM
    brons_beschermde_stads_en_dorpsgezichten.townscapes x
    JOIN
    brons_rijksmonumentenregister.tblTEXT_OBJECT y
    ON
    x.bron_id = y.OBJ_NUMMER
    """
).fetchdf()
df

Maak een kaartje met de stads- en dorpsgezichten van Amersfoort, met daarbinnen punten van gebouwen die ook rijksmonumenten zijn (zelfde testje als Stan)

In [ ]:
stadsgezichten_amersfoort = duckdb.sql("""
SELECT ST_asText(geometry) as geometry,naam
FROM brons_beschermde_stads_en_dorpsgezichten.townscapes
WHERE ST_Within(
    geometry,
    (
        SELECT geometry
        FROM brons_bestuurlijke_gebieden_2026.gemeentegebied
        WHERE naam = 'Amersfoort'
    )
);""").fetchdf()
stadsgezichten_amersfoort
stadsgezichten_amersfoort['geometry'] = stadsgezichten_amersfoort['geometry'].apply(wkt.loads)
stadsgezichten_amersfoort = gpd.GeoDataFrame(stadsgezichten_amersfoort, geometry='geometry')


In [ ]:
rijksmon_in_stad_amersfoort = duckdb.sql("""
SELECT ST_asText(geometry) as geometry,rijksmonument_nummer
FROM brons_rijksmonumentenpunten.rijksmonumentenpunten
WHERE ST_Within(geometry,
    (SELECT ST_collect(list(geometry))
        FROM brons_beschermde_stads_en_dorpsgezichten.townscapes
        WHERE ST_Within(
            geometry,
                (
                SELECT geometry
                FROM brons_bestuurlijke_gebieden_2026.gemeentegebied
                WHERE naam = 'Amersfoort'
                )
            )
        )
);""").fetchdf()
rijksmon_in_stad_amersfoort['geometry'] = rijksmon_in_stad_amersfoort['geometry'].apply(wkt.loads)
rijksmon_in_stad_amersfoort = gpd.GeoDataFrame(rijksmon_in_stad_amersfoort, geometry='geometry')
rijksmon_in_stad_amersfoort

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
stadsgezichten_amersfoort.explore(ax=ax)
rijksmon_in_stad_amersfoort.explore(ax = ax, color='red')
rijksmon_in_stad_amersfoort.apply(lambda x: ax.annotate(text=x['rijksmonument_nummer'], xy=x.geometry.centroid.coords[0], ha='center'), axis=1)

sla de hele database op (als parquet, maar includief load.sql en schema.sql)
let op, kan even duren (als je bijvoorbeeld de BAG ook open hebt staan...)

In [ ]:
duckdb.sql("SHOW SCHEMAS;") # check of ze weg zijn


laad die hele database weer in (kan eventjes duren, wederom, vooral als je het BAG hebt ingeladen)

# Pas de datakwaliteits richtlijnen toe



In [ ]:
dimensies_voldaan = 0

## Dimensie 1: referentiële integriteit en unieke identificatie
- Welke kolom bevat de ID?
    - Is deze ID uniek?
    - Is (als dit te bepalen is/de ingeladen bron het origineel is) in de originele database/bestandsstructuur vastgesteld dat dit een uniek veld is en er geen nullwaarden mogen zijn?
- Welke kolom(men) bevat(ten) foreign key(s)?
    - Welke dataset verwijst de foreign key naar en klopt deze identifier?
    - Hebben alle foreign keys een eigen kolom?

### functies dimensie 1

In [ ]:
def id_column_compliant(id_column, table_name):
    """Bepaalt of de id kolom voldoet aan de voorwaarden van dimensie 1"""
    return True
    pass


def foreign_key_compliant(id_column, id_column_foreign, table_name, foreign_table_name):
    """Bepaalt of het gebruik van foreign keys voldoet aan de voorwaarden van dimensie 1"""
    return False
    pass



### Stads- en dorpsgezichten

In [11]:
# beschermde_stads_en_dorpsgezichten in brons
duckdb.sql("""
           DESCRIBE  brons_beschermde_stads_en_dorpsgezichten.townscapes;
           """)
id_column = "bron_id"
table_name = "brons_beschermde_stads_en_dorpsgezichten.townscapes"
dict_foreign_key = None


┌─────────────┬────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │      column_type       │  null   │   key   │ default │  extra  │
│   varchar   │        varchar         │ varchar │ varchar │ varchar │ varchar │
├─────────────┼────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ bron_id     │ VARCHAR                │ YES     │ NULL    │ NULL    │ NULL    │
│ naam        │ VARCHAR                │ YES     │ NULL    │ NULL    │ NULL    │
│ in_procedu  │ TIMESTAMP              │ YES     │ NULL    │ NULL    │ NULL    │
│ aangewezen  │ TIMESTAMP              │ YES     │ NULL    │ NULL    │ NULL    │
│ jurstatus   │ VARCHAR                │ YES     │ NULL    │ NULL    │ NULL    │
│ datum_begr  │ TIMESTAMP              │ YES     │ NULL    │ NULL    │ NULL    │
│ ondergrond  │ VARCHAR                │ YES     │ NULL    │ NULL    │ NULL    │
│ opperv_ha   │ DOUBLE                 │ YES     │ NULL    │ NULL    │ NULL    │
│ opperv_km2  │ DOUBLE      

In [ ]:
if id_column_compliant(id_column, table_name):
    if dict_foreign_key != None:
        for key in dict_foreign_key:
            if foreign_key_compliant(id_column, key, key, dict_foreign_key(key)):
                dim1 = True
            else:
                print("foreign keys voldoet niet")
                dim1 = False
                break
    else:
        dim1 = True
else:
    print("id voldoet niet")
    dim1 = False

if dim1:
    dimensies_voldaan += 1


## Dimensie 2: Structurere consistentie (logische opbouw)

## Dimensie 3: Waardenvaliditeit (inhoudelijke juistheid)

## Dimensie 4: Granulariteit en eenduidige representatie

## Dimensie 5: Semantische eenduidigheid en interoperabiliteit

## Dimensie 6: Ruimtelijke referentie eenduidigheid

## Dimensie 7 Ruimtelijke schaal- en resolutieconsistentie

## Dimensie 8: Ruimtelijke geometrische correctheid

## Dimensie 9: Temporele correctheid (definitie in progress)